# Proyecto 3 · Parte 3 en Google Colab: CPU vs CUDA con el mismo código

Este notebook repite en Colab **el mismo experimento** que `scripts/parte3_benchmark.sh` en la laptop RTX 3060:
mismas imágenes, particiones, modelo (CNN base, 1,145,693 parámetros), Adam 0.001, lote 64, 20 épocas y float32 estricto.
Sirve como segundo entorno para comparar CUDA en otra GPU (normalmente una NVIDIA T4).

**Flujo, separado igual que en el repositorio**

| Fase | Dónde corre | Módulo |
|---|---|---|
| Inventario, lectura JPEG, resize, RGB y exportación NPZ | CPU, 1 o varios procesos | `asl_cpu.prepare` / `asl_cpu.data` |
| Carga del NPZ, normalización y `tf.data` | CPU | `asl_cpu.train` |
| Forward, pérdida, gradientes y Adam | CPU (referencia) o GPU con CUDA | `asl_cpu.train --device cpu/gpu`, `--optimizer numba` |
| Repeticiones, tiempos, speedup y gráficos | CPU | `asl_cpu.benchmark`, `asl_cpu.analyze_benchmark` |

**Reglas de comparación.** El speedup se calcula **dentro de Colab** (CPU de Colab contra GPU de Colab). Comparar la GPU de
Colab con la de la laptop compara hardware con el mismo código; nunca se mezcla la CPU de un equipo con la GPU de otro.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*. Todo lo pesado corre en
procesos aparte dentro de un entorno virtual propio, así que el kernel del notebook no importa TensorFlow ni cambia sus
librerías. Los resultados se copian a Google Drive después de cada corrida: si Colab se desconecta, vuelva a ejecutar
las celdas y el experimento continúa donde quedó.

Duración aproximada: instalación y datos 10 min; preparación 10 min; GPU ~15 min; **CPU de Colab (2 vCPU) ~30–45 min por
corrida**. Por eso la CPU usa 3 repeticiones por defecto (`CPU_REPS`); súbalo a 5 si el tiempo de la sesión lo permite.

In [ ]:
# ---- Parámetros del experimento (ajuste solo esta celda) ----
DRIVE_DIR = "/content/drive/MyDrive/Proyecto3_Paralela"   # caché del dataset y copia de los resultados
REPO_URL = "https://github.com/Teviets/Proyecto-3---Paralela.git"  # si el repo es privado, use REPO_ZIP
REPO_ZIP = DRIVE_DIR + "/Proyecto-3---Paralela-main.zip"   # alternativa: ZIP de 'Code → Download ZIP' subido a Drive
WORK = "/content/Proyecto-3---Paralela-main"
GPU_REPS = 5          # repeticiones medidas de cada versión GPU
CPU_REPS = 3          # repeticiones medidas de la referencia CPU (5 si hay tiempo)
PREP_REPS = 5         # repeticiones de la preparación con 1 y con N procesos
COOLDOWN_S = 20       # pausa entre corridas
VENV = "/content/venv"
PY = VENV + "/bin/python"

In [ ]:
# 1. Verificar que el entorno tiene GPU NVIDIA
import subprocess, shutil
assert shutil.which("nvidia-smi"), "Este entorno no tiene GPU. Cambie el tipo de entorno de ejecución a GPU."
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
GPU_NAME = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip().splitlines()[0]
LABEL = "colab_" + GPU_NAME.replace("NVIDIA", "").replace("Tesla", "").strip().split()[0].lower()
print("Etiqueta del entorno:", LABEL)

In [ ]:
# 2. Montar Google Drive (caché del dataset y copia de seguridad de los resultados)
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# 3. Obtener el código del repositorio
import os, subprocess, zipfile, glob
if not os.path.isdir(WORK):
    if os.path.isfile(REPO_ZIP):
        with zipfile.ZipFile(REPO_ZIP) as z:
            z.extractall("/content")
        found = [d for d in glob.glob("/content/*/asl_cpu") if "venv" not in d]
        if found and os.path.dirname(found[0]) != WORK:
            os.rename(os.path.dirname(found[0]), WORK)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
os.chdir(WORK)
print(os.getcwd())
print(sorted(os.listdir(".")))
assert os.path.isfile("asl_cpu/benchmark.py"), "Falta la parte 3 en el repositorio (asl_cpu/benchmark.py)."

In [ ]:
# 4. Entorno virtual con las versiones fijadas del proyecto (no modifica las librerías del notebook)
import os
if not os.path.isfile(PY):
    !pip -q install virtualenv
    !virtualenv -q {VENV}
!{PY} -m pip -q install --upgrade pip
!{PY} -m pip -q install -r requirements-gpu.txt
!{PY} -m pip check || true
!{PY} -c "import sys, tensorflow as tf, numba, cupy; print(sys.version.split()[0], 'TF', tf.__version__, 'numba', numba.__version__, 'cupy', cupy.__version__)"

In [ ]:
# 5. Dataset ASL Alphabet (Kaggle, endpoint público) con caché en Drive
import os, subprocess
zip_drive = DRIVE_DIR + "/asl-alphabet.zip"
if not os.path.isfile(zip_drive):
    subprocess.run(["curl", "-L", "--fail", "-o", zip_drive,
                    "https://www.kaggle.com/api/v1/datasets/download/grassknoted/asl-alphabet"], check=True)
if not os.path.isdir("data/raw/asl_alphabet_train"):
    os.makedirs("data/raw", exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", zip_drive, "-d", "data/raw"], check=True)
count = sum(len(files) for _, _, files in os.walk("data/raw/asl_alphabet_train"))
print("Imágenes de entrenamiento:", count)
assert count == 87000, "Se esperaban 29 clases × 3,000 imágenes."

In [ ]:
# 6. Registrar el entorno CUDA y probar matmul/convolución en /GPU:0
OUT = f"results/parte3/{LABEL}"
MIRROR = f"{DRIVE_DIR}/results/parte3/{LABEL}"
import os
os.makedirs(OUT, exist_ok=True)
!{PY} -m asl_cpu.gpu_check --output {OUT}/gpu_environment.json | tail -n 15
!nvidia-smi > {OUT}/nvidia_smi_inicio.txt

In [ ]:
# 7. Preparar los datos de entrenamiento (CPU, todos los núcleos) y comprobar que son los MISMOS que en la laptop
import json, os
if not os.path.isfile("data/processed/baseline/metadata.json"):
    !{PY} -X utf8 -m asl_cpu.prepare --workers {os.cpu_count()} --output-dir data/processed/baseline
mine = json.load(open("data/processed/baseline/metadata.json"))
reference = json.load(open("results/preparation_sequential_01/metadata.json"))   # preparación de la parte 1
same_arrays = mine["array_sha256"] == reference["array_sha256"]
same_partition = mine["partition_sha256"] == reference["partition_sha256"]
print("Mismas particiones que la referencia del equipo:", same_partition)
print("Mismos arreglos (bit a bit) que la referencia del equipo:", same_arrays)
if not same_arrays:
    print("AVISO: los píxeles difieren (versión de OpenCV/libjpeg distinta). La comparación CPU-GPU dentro de Colab "
          "sigue siendo válida, pero regístrelo al comparar entre entornos.")

In [ ]:
# 8. Pruebas rápidas antes de medir: unidades del proyecto y una corrida smoke de cada configuración (~3 min)
!{PY} -m unittest tests.test_benchmark tests.test_model tests.test_gpu_device tests.test_cuda_kernels 2>&1 | tail -n 4
!{PY} -m asl_cpu.benchmark --output-dir results/parte3/{LABEL}_smoke --label {LABEL}_smoke --smoke --reps 1 --warmup none --threads {os.cpu_count()} --prepare-workers "" --cooldown 0 --quiet

## Experimento medido

La celda siguiente lanza el experimento **en segundo plano** y la siguiente lo vigila. Así puede tomar las
**capturas de pantalla** que pide el enunciado mientras corre:

1. Durante una corrida **CPU** (`train/cpu/...`): la tabla de núcleos debe mostrar los 2 vCPU ocupados y `nvidia-smi` sin
   proceso de Python.
2. Durante una corrida **GPU** (`train/gpu_keras/...` y `train/gpu_numba/...`): `nvidia-smi` con el proceso de Python,
   memoria asignada y utilización alta.
3. El panel **Recursos** de Colab (arriba a la derecha, RAM / GPU RAM) en ambos casos.

Si Colab se desconecta, vuelva a ejecutar las celdas 1–7 y después esta: el orquestador restaura lo que ya está en Drive
y continúa.

In [ ]:
# 9. Lanzar el experimento en segundo plano (reanudable, con copia a Drive después de cada corrida)
import os, subprocess
LOG = f"{OUT}/benchmark_console.log"
cmd = [PY, "-m", "asl_cpu.benchmark", "--output-dir", OUT, "--label", LABEL,
       "--reps", str(GPU_REPS), "--reps-override", f"cpu={CPU_REPS}", "--threads", str(os.cpu_count()),
       "--prepare-workers", f"1,{os.cpu_count()}", "--prepare-reps", str(PREP_REPS), "--prepare-warmup", "1",
       "--warmup", "smoke", "--cooldown", str(COOLDOWN_S), "--mirror-dir", MIRROR]
log = open(LOG, "a")
job = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("PID", job.pid)
print(" ".join(cmd))

In [ ]:
# 10. Monitor en vivo hasta que termine (CPU por núcleo, nvidia-smi y avance). Buen momento para las capturas.
import json, subprocess, time
from IPython.display import clear_output

def per_core():
    out = subprocess.run([PY, "-c", "import psutil; print(psutil.cpu_percent(percpu=True, interval=1))"],
                         capture_output=True, text=True).stdout.strip()
    return out

while job.poll() is None:
    try:
        runs = json.load(open(f"{OUT}/manifest.json"))["runs"]
        done = sum(r.get("status") == "complete" for r in runs)
        current = [r["name"] for r in runs if r.get("status") == "running"]
    except Exception:
        done, current = 0, []
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    tail = subprocess.run(["tail", "-n", "6", LOG], capture_output=True, text=True).stdout
    clear_output(wait=True)
    print(time.strftime("%H:%M:%S"), f"| corridas completas: {done} | en curso: {current}")
    print("CPU por procesador lógico (%):", per_core())
    print(smi)
    print(tail)
    time.sleep(8)
print("Terminó con código", job.returncode)
!tail -n 5 {LOG}

In [ ]:
# 11. Análisis: tablas, speedup por fase, Amdahl, recursos y gráficos
!{PY} -m asl_cpu.analyze_benchmark {OUT}
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{OUT}/analisis/RESULTADOS.md", encoding="utf-8").read().split("## Gráficos")[0]))
import glob
for png in sorted(glob.glob(f"{OUT}/analisis/*.png")):
    display(Image(png))

In [ ]:
# 12. Comparación con la laptop RTX 3060 (si sus resultados ya están en el repositorio)
import os
if os.path.isfile("results/parte3/rtx3060/manifest.json"):
    !{PY} -m asl_cpu.analyze_benchmark results/parte3/rtx3060 {OUT} --output-dir results/parte3/comparacion
    from IPython.display import Markdown, Image, display
    display(Markdown(open("results/parte3/comparacion/COMPARACION.md", encoding="utf-8").read()))
    display(Image("results/parte3/comparacion/comparacion_entornos.png"))
else:
    print("Aún no están los resultados de la laptop (results/parte3/rtx3060). La comparación se puede generar después "
          "en cualquier equipo con: python -m asl_cpu.analyze_benchmark results/parte3/rtx3060 "
          f"results/parte3/{LABEL} --output-dir results/parte3/comparacion")

In [ ]:
# 13. Empaquetar los resultados para subirlos al repositorio (sin pesos ni predicciones, que Git excluye)
import os, shutil, zipfile
name = f"resultados_parte3_{LABEL}.zip"
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [OUT, "results/parte3/comparacion"]:
        for root, _, files in os.walk(folder):
            for f in files:
                if f.endswith((".weights.h5", "predictions.npz")):
                    continue
                z.write(os.path.join(root, f))
shutil.copy(name, DRIVE_DIR)
print("Guardado en Drive:", os.path.join(DRIVE_DIR, name))
from google.colab import files
files.download(name)

## Anexo: el código de cada fase

Las celdas anteriores ejecutan los módulos del repositorio. Esta celda muestra, sin importarlos, las funciones que
separan el flujo: preparación en CPU, colocación del modelo en el dispositivo y el kernel CUDA propio.

In [ ]:
import ast
def show(path, *names):
    source = open(path, encoding="utf-8").read()
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names:
            print(f"# ---- {path} · {node.name}\n" + ast.get_source_segment(source, node) + "\n")

show("asl_cpu/data.py", "read_image", "prepare_arrays")          # Fase 1: CPU, multiproceso
show("asl_cpu/train.py", "resolve_training_device", "sync_device") # Fase 2: elección de dispositivo y sincronización
show("asl_cpu/cuda_kernels.py", "adam_update", "adam_launch_config") # Kernel CUDA propio (Numba)